# Size and speed

How small and how fast the released simulators are. Python numbers come from `eval/latency.py` (onnxruntime, CPU). Browser numbers come from `eval/browser_bench/` (onnxruntime-web 1.22, the same runtime as the deployed demo, closed loop, batch 1, headless Chromium). All on one Apple M1 laptop.

In [1]:
import json, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from eval import plotstyle as ps

ps.apply()
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)
pd.set_option("display.precision", 4)


In [2]:
manifest = json.loads(Path("../artifacts/MANIFEST.json").read_text())
rows = []
for path, m in manifest.items():
    if path.endswith(".onnx"):
        rows.append({"model": path.replace("artifacts/", ""), "ONNX values (weights + shape constants)": m["params"], "fp32 size (MB)": m["bytes"] / 1e6})
sizes = pd.DataFrame(rows)
sizes.loc[len(sizes)] = {"model": "CARLA total", "ONNX values (weights + shape constants)": sizes[sizes.model.str.startswith("carla")]["ONNX values (weights + shape constants)"].sum(),
                         "fp32 size (MB)": sizes[sizes.model.str.startswith("carla")]["fp32 size (MB)"].sum()}
sizes.loc[len(sizes)] = {"model": "TinySkies total", "ONNX values (weights + shape constants)": sizes[sizes.model.str.startswith("tinyskies/")]["ONNX values (weights + shape constants)"].sum(),
                         "fp32 size (MB)": sizes[sizes.model.str.startswith("tinyskies/")]["fp32 size (MB)"].sum()}
sizes.to_csv(FIGURES / "model_sizes.csv", index=False)
sizes

,model,ONNX values (weights + shape constants),fp32 size (MB)
0,carla/world_transition.onnx,1551854,6.2612
1,carla/lidar_encoder.onnx,27462,0.1160
2,tinyskies/encoder.onnx,660480,2.6480
3,tinyskies/decoder.onnx,1282933,5.1613
4,tinyskies/latent_sim.onnx,1842071,7.4077
5,CARLA total,1579316,6.3772
6,TinySkies total,3785484,15.2171


In [3]:
py = json.loads((Path("eval") / "latency_python.json").read_text())
rows = []
for threads in ("threads_1", "threads_4"):
    for name, r in py[threads].items():
        rows.append({"runtime": f"python onnxruntime ({threads.replace('_', ' ')})", "measurement": name,
                     "median ms": r["median_ms"], "p90 ms": r["p90_ms"], "max steps/s": 1000 / r["median_ms"]})
br = json.loads((Path("eval") / "latency_browser.json").read_text())
for r in br["results"]:
    for key, label in (("carla_step", "carla_step (transition)"), ("tinyskies_step", "tinyskies_step (latent_sim + decoder)")):
        if key in r:
            rows.append({"runtime": f"browser {r['ep']} ({r['threads']} thread{'s' if r['threads'] > 1 else ''})",
                         "measurement": label, "median ms": r[key]["median_ms"], "p90 ms": r[key]["p90_ms"],
                         "max steps/s": 1000 / r[key]["median_ms"]})
        elif "error" in r:
            rows.append({"runtime": f"browser {r['ep']}", "measurement": label, "median ms": np.nan, "p90 ms": np.nan, "max steps/s": np.nan})
latency = pd.DataFrame(rows)
latency.to_csv(FIGURES / "latency_table.csv", index=False)
print(py["machine"], br["machine"])
latency

{'cpu': 'Apple M1', 'onnxruntime': '1.30.0', 'python': '3.11.13'} {'cpu': 'Apple M1', 'cores': 8, 'platform': 'darwin 25.2.0'}


,runtime,measurement,median ms,p90 ms,max steps/s
0,python onnxruntime (threads 1),carla_step (transition),6.088,6.219,164.2576
1,python onnxruntime (threads 1),carla_reset (lidar_encoder),0.130,0.132,7692.3077
2,python onnxruntime (threads 1),tinyskies_step (latent_sim + decoder),23.734,25.402,42.1336
3,python onnxruntime (threads 1),tinyskies_latent_sim,0.501,0.644,1996.0080
4,python onnxruntime (threads 1),tinyskies_decoder,23.492,24.302,42.5677
5,python onnxruntime (threads 1),tinyskies_reset (encoder),1.576,1.637,634.5178
6,python onnxruntime (threads 4),carla_step (transition),2.569,3.000,389.2565
7,python onnxruntime (threads 4),carla_reset (lidar_encoder),0.070,0.144,14285.7143
8,python onnxruntime (threads 4),tinyskies_step (latent_sim + decoder),15.610,19.586,64.0615
9,python onnxruntime (threads 4),tinyskies_latent_sim,0.354,0.369,2824.8588
